In [205]:
import json
from pathlib import Path
import sys
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve().parent

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Calculating weights for positions without any data
For the following positions: CAM, Wide Midfielder, Wingback, we have no available data to calculate weights with. This means that we cannot run them through the same process as other positions: pre-processing and PCA. Creating synthetic data for these positions is also a challenging process, and one that may not yield accurate results.<br><br>
Instead, we will adapt the weights of similar positions:
 - For CAM, we will use the weights of the Central Midfielder (CM) position, as they share similar responsibilities on the field.
 - For Wide Midfielder, we will use the weights of the Winger (RW/LW) position, as they both operate on the flanks and have similar roles in attacking.
 - For Wingback, we will use the weights of the Fullback (RB/LB) position, as they both have defensive duties while also contributing to the attack on the wings.<br><br>

We will take in the relevant proxy weights, and adjust them using multipliers to make the weights more relevant to the specific position and the standard tactical expectations for that position. These multipliers will be determined based on the typical responsibilities and contributions of each position on the field. For example, for CAM, we may increase the weights for passing and xT, while for Wingback, we may increase the weights for attacking contributions and and distance covered.<br><br>

This method is composed of a few main steps:
1. Defining the weight multipliers for each position based on their typical responsibilities and contributions on the field.
2. Applying these multipliers to the proxy weights from the similar positions to calculate the adjusted weights for CAM, Wide Midfielder, and Wingback, and then re-normalizing the weights so that they sum to 1.
3. Creating estimates for the means and standard deviations for each stat for these positions, based on the data we have for similar positions and any relevant football knowledge or assumptions we can make about these positions.

This approach allows us to estimate the weights for these positions in a way that is informed by the data we have for similar positions, while also accounting for the unique characteristics of each position. It is important to note that this method is not perfect and may not capture all the nuances of each position, but it provides a reasonable approximation given the lack of direct data.

### The wingback
**Proxy: Fullback (RB/LB)**<br><br>
Wingbacks run more, dribble more, and defend less than traditional fullbacks. Therefore, we will increase the weights for distance covered, dribbles, and attacking contributions, while decreasing the weights for defensive actions. The specific multipliers will be determined based on the typical responsibilities of wingbacks in modern football tactics.

**Step 1: Define weight multipliers for Wingback**
 - `Goals`: **2.0** (wingbacks should be much more rewarded for goals than traditional fullbacks)
 - `Assists`: **1.5** (similar to goals, wingbacks should be more rewarded for assists than traditional fullbacks)
 - `Shots`: **1.5** (wingbacks should be rewarded more for shots than traditional fullbacks due to their increased attacking involvement)
 - `Shot Accuracy`: **1.5** (wingbacks should have better shot accuracy due to more attacking involvement)
 - `Passes`: **1.0** (wingbacks may have similar passing involvement as fullbacks)
 - `Pass Accuracy`: **1.0** (wingbacks may have similar pass accuracy as fullbacks)
 - `Dribbles`: **1.8** (taking players on is much more important for wingbacks than traditional fullbacks)
 - `Dribble Success Rate`: **1.8** (wingbacks should be taking players on more effectively than traditional fullbacks)
 - `Tackles`: **0.6** (wingbacks have less defensive responsibilities)
 - `Tackle Success Rate`: **0.7** (due to their reduced defensive responsibilities, wingbacks may have a lower tackle success rate than traditional fullbacks)
 - `Offsides`: **1.0** (wingbacks may have similar offsides involvement as fullbacks)
 - `Fouls Committed`: **1.0** (wingbacks may have similar fouls committed as fullbacks)
 - `Possession Won`: **0.6** (there is a much lower defensive burden on wingbacks, so they may win fewer possessions than traditional fullbacks)
 - `Possession Lost`: **0.7** (as wingbacks are involved in more attacking play, they should be punished less for losing possessions)
 - `Distance Covered`: **1.5** (As wingbacks have more attacking responsibilities, they should be covering more distance than traditional fullbacks)
 - `Distance Sprinted`: **1.5** (similar to distance covered, wingbacks should be sprinting more than traditional fullbacks due to their increased involvement in attacking plays)
 - `xT`: **1.8** (wingbacks should be contributing more to expected threat than traditional fullbacks due to their increased attacking involvement)

In [206]:
wb_multipliers = {
    'goals_p90': 2.0,
    'assists_p90': 8.0,
    'non_goal_shots_p90': 1.5,
    'shot_accuracy': 1.5,
    'passes_p90': 1.2,
    'pass_accuracy': 0.9,
    'dribbles_p90': 1.8,
    'dribble_success_rate': 1.8,
    'tackles_p90': 0.8,
    'tackle_success_rate': 0.8,
    'offsides_p90': 0.6,
    'fouls_committed_p90': 0.7,
    'possession_won_p90': 0.6,
    'possession_lost_p90': 0.6,
    'distance_covered_p90': 1.7,
    'distance_sprinted_p90': 1.7,
    'xt_bonus_p90': 2.0
}

**Step 2: Apply multipliers to proxy weights and re-normalizing**<br><br>
We will take the weights from the Fullback position and apply the defined multipliers to calculate the adjusted weights for the Wingback position. After applying the multipliers, we will re-normalize the weights to ensure they sum up to 1.

In [207]:
# Importing fullback weights
col_names = ["goals_p90", "assists_p90", "non_goal_shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
position_weights_path = project_root / "workshop" / "ratings_creation" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["RB"]
fullback_weights = {col: weights_dict[col] for col in col_names}

In [208]:
wingback_weights = {}
total_wb_weight = 0

for stat, base_weight in fullback_weights.items():
    multiplier = wb_multipliers.get(stat, 1.0) # Default to 1.0 if not listed
    raw_new_weight = base_weight * multiplier
    wingback_weights[stat] = raw_new_weight
    total_wb_weight += raw_new_weight

# Re-normalize so they sum to 1.0
wingback_weights = {k: v/total_wb_weight for k, v in wingback_weights.items()}

In [209]:
print("Wingback Weights:")
for stat, weight in wingback_weights.items():
    print(f"{stat}: {weight:.4f}")

Wingback Weights:
goals_p90: 0.0000
assists_p90: 0.1636
non_goal_shots_p90: 0.0062
shot_accuracy: 0.0001
passes_p90: 0.1031
pass_accuracy: 0.0336
dribbles_p90: 0.1395
dribble_success_rate: 0.0384
tackles_p90: 0.0732
tackle_success_rate: 0.0476
offsides_p90: 0.0002
fouls_committed_p90: 0.0248
possession_won_p90: 0.0498
possession_lost_p90: 0.0378
distance_covered_p90: 0.0832
distance_sprinted_p90: 0.0841
xt_bonus_p90: 0.1149


In [210]:
# Saving these weights back to the position_weights.json file
section_names = ["RWB", "LWB"]
weights_dict = dict(zip(col_names, wingback_weights.values()))
position_weights_path = project_root / "workshop" / "ratings_creation" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

for section_name in section_names:
    position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

**Step 3: Estimate means and standard deviations for Wingback**<br><br>

To estimate the means and standard deviations for each stat for the Wingback position, we use a similar method to the weights, where we take the means and standard deviations from the Fullback position and adjust them based on the expected differences in performance for wingbacks compared to fullbacks. For example, we would expect wingbacks to have higher means for attacking stats like goals, assists, and xT, and lower means for defensive stats like tackles and possession won. The specific adjustments would be based on football knowledge and assumptions about the typical performance of wingbacks compared to fullbacks.

In [211]:
wb_stat_multipliers = {
    'goals_p90': 2.0, 
    'assists_p90': 2.4, 
    'non_goal_shots_p90': 1.5, 
    'shot_accuracy': 4.0,
    'dribbles_p90': 1.4,
    'dribble_success_rate': 0.8,
    'tackles_p90': 0.6, 
    'tackle_success_rate': 0.9,
    'offsides_p90': 2.0,
    'possession_won_p90': 0.7,
    'possession_lost_p90': 2.0,
    'distance_covered_p90': 1.15, 
    'distance_sprinted_p90': 1.3,
    'xt_bonus_p90': 1.5
}

In [212]:
# Importing the fullback means and standard deviations
position_means_stds_path = project_root / "workshop" / "ratings_creation" / "position_means_stds.json"
with open(position_means_stds_path, 'r') as f:
    position_means_stds = json.load(f)
fullback_means_stds = position_means_stds["RB"]

In [213]:
for stat, stats in fullback_means_stds.items():
    print(f"{stat}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

goals_p90: mean=0.0000, std=0.2500
assists_p90: mean=0.0468, std=0.1571
non_goal_shots_p90: mean=0.0414, std=0.1502
shot_accuracy: mean=3.0000, std=16.5489
passes_p90: mean=18.9046, std=5.9164
pass_accuracy: mean=88.4852, std=12.0395
dribbles_p90: mean=13.8176, std=4.7999
dribble_success_rate: mean=88.2148, std=20.3804
tackles_p90: mean=7.3457, std=2.3203
tackle_success_rate: mean=45.3185, std=28.6873
offsides_p90: mean=0.0078, std=0.0518
fouls_committed_p90: mean=0.2469, std=0.2791
possession_won_p90: mean=1.7767, std=0.4080
possession_lost_p90: mean=0.9643, std=0.4754
distance_covered_p90: mean=11.4174, std=1.4298
distance_sprinted_p90: mean=4.7954, std=1.7028
xt_bonus_p90: mean=1.4634, std=0.5441


In [214]:
wingback_means_stds = {}
for stat, stats in fullback_means_stds.items():
    mean_multiplier = wb_stat_multipliers.get(stat, 1.0)
    std_multiplier = np.sqrt(mean_multiplier) # Assuming variance scales with mean
    wingback_means_stds[stat] = {
        "mean": stats["mean"] * mean_multiplier,
        "std": stats["std"] * std_multiplier
    }
    # manually setting the mean and std for goals_p90
    if stat == "goals_p90":
        wingback_means_stds[stat]["mean"] = 0.08
        wingback_means_stds[stat]["std"] = 0.15
    elif stat == "shot_accuracy":
        # Override the crushed variance inherited from the Fullbacks
        wingback_means_stds[stat]["std"] = 18.5

In [215]:
print("Wingback Means and Standard Deviations:")
for stat, stats in wingback_means_stds.items():
    print(f"{stat}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

Wingback Means and Standard Deviations:
goals_p90: mean=0.0800, std=0.1500
assists_p90: mean=0.1124, std=0.2433
non_goal_shots_p90: mean=0.0621, std=0.1839
shot_accuracy: mean=12.0000, std=18.5000
passes_p90: mean=18.9046, std=5.9164
pass_accuracy: mean=88.4852, std=12.0395
dribbles_p90: mean=19.3446, std=5.6794
dribble_success_rate: mean=70.5719, std=18.2287
tackles_p90: mean=4.4074, std=1.7973
tackle_success_rate: mean=40.7867, std=27.2152
offsides_p90: mean=0.0155, std=0.0732
fouls_committed_p90: mean=0.2469, std=0.2791
possession_won_p90: mean=1.2437, std=0.3413
possession_lost_p90: mean=1.9286, std=0.6724
distance_covered_p90: mean=13.1301, std=1.5333
distance_sprinted_p90: mean=6.2340, std=1.9415
xt_bonus_p90: mean=2.1951, std=0.6664


In [216]:
# Storing the new means and standard deviations back to the position_means_stds.json file
section_names = ["RWB", "LWB"]
position_means_stds_path = project_root / "workshop" / "ratings_creation" / "position_means_stds.json"
if position_means_stds_path.exists():
    with open(position_means_stds_path, "r") as f:
        position_means_stds = json.load(f)
else:
    position_means_stds = {}

for section_name in section_names:
    position_means_stds[section_name] = wingback_means_stds

with open(position_means_stds_path, "w") as f:
    json.dump(position_means_stds, f, indent=4)

## The Wide Midfielder
**Proxy: Winger (RW/LW)**<br><br>

Wide Midfielders shoot way less than Wingers, cross more, and actually have to track back and defend. Therefore, we will decrease the weights for goals and shots, increase the weights for crossing and defensive actions, and adjust the weights for passing and dribbling to reflect the different responsibilities of Wide Midfielders compared to Wingers. The specific multipliers will be determined based on the typical responsibilities of Wide Midfielders in modern football

**Step 1: Define weight multipliers for Wide Midfielder**
 - `Goals`: **0.8** (Wide Midfielders shouldn't be expected to score as much as Wingers, as they often play deeper and have more defensive responsibilities)
 - `Assists`: **1.2** (Wide Midfielders should be expected to contribute creatively a bit more than wingers, as they often have more involvement in build-up play and crossing)
 - `Shots`: **0.5** (Wide Midfielders should be expected to take a lot less shots than wingers)
 - `Shot Accuracy`: **0.5** (Wide Midfielders should be expected to have a lower shot accuracy than wingers, as they take fewer shots and often from less dangerous positions)
 - `Passes`: **1.5** (Wide Midfielders should be expected to have more passing involvement than Wingers, as they often play deeper and are more involved in build-up play)
 - `Pass Accuracy`: **1.5** (Wide Midfielders should be expected to have a higher pass accuracy than Wingers, as they are often making their passes from deeper positions and may have more time on the ball)
 - `Dribbles`: **0.8** (Wide Midfielders should be expected to dribble less than wingers, less hero balling and more efficient dribbling)
 - `Dribble Success Rate`: **1.0** (Wide Midfielders may have a similar dribble success rate as Wingers, but this can vary greatly depending on the player's style and role)
 - `Tackles`: **8.0** (Wide Midfielders are typically expected to track back and help out their fullback much more than wingers, so they should be expected to make more tackles)
 - `Tackle Success Rate`: **6.0** (Wide Midfielders should be expected to have a higher tackle success rate than Wingers, as they are often making more tackles and may be more defensively focused)
 - `Offsides`: **1.0** (Wide Midfielders may have similar offsides involvement as Wingers, but this can vary greatly depending on the player's role and style)
 - `Fouls Committed`: **1.0** (Wide Midfielders may have similar fouls committed as Wingers, but this can vary greatly depending on the player's role and style)
 - `Possession Won`: **7.0** (Wide Midfielders should be expected to win more possessions than Wingers, as they often have more defensive responsibilities)
 - `Possession Lost`: **1.5** (Wide Midfielders should be punished more for losing possession in midfield)
 - `Distance Covered`: **1.1** (As Wide Midfielders have more defensive responsibilities than Wingers, they may be expected to cover slightly more distance than Wingers, but this can vary greatly depending on the player's role and style)
 - `Distance Sprinted`: **1.1** (similar to distance covered, Wide Midfielders may be expected to sprint slightly more than Wingers due to their increased defensive responsibilities, but this can vary greatly depending on the player's role and style)
 - `xT`: **1.3** (A higher reward for contributing expected threat, often from wider or deeper positions)

In [217]:
wm_multipliers = {
    'goals_p90': 0.8,
    'assists_p90': 1.2,
    'non_goal_shots_p90': 0.6,
    'shot_accuracy': 0.6,
    'passes_p90': 2.0,
    'pass_accuracy': 2.5,
    'dribbles_p90': 0.9,
    'dribble_success_rate': 1.2,
    'tackles_p90': 3.0,
    'tackle_success_rate': 3.0,
    'offsides_p90': 1.0,
    'fouls_committed_p90': 1.0,
    'possession_won_p90': 3.0,
    'possession_lost_p90': 2.5,
    'distance_covered_p90': 1.1,
    'distance_sprinted_p90': 1.1,
    'xt_bonus_p90': 1.1
}

**Step 2: Apply multipliers to proxy weights and re-normalizing**<br><br>
We will take the weights from the Winger position and apply the defined multipliers to calculate the adjusted weights for the Wide Midfielder position. After applying the multipliers, we will re-normalize the weights to ensure they sum up to 1.

In [218]:
# Importing winger weights
col_names = ["goals_p90", "assists_p90", "non_goal_shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
position_weights_path = project_root / "workshop" / "ratings_creation" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["RW"]
winger_weights = {col: weights_dict[col] for col in col_names}

In [219]:
wide_midfielder_weights = {}
total_wm_weight = 0

for stat, base_weight in winger_weights.items():
    multiplier = wm_multipliers.get(stat, 1.0) # Default to 1.0 if not listed
    raw_new_weight = base_weight * multiplier
    wide_midfielder_weights[stat] = raw_new_weight
    total_wm_weight += raw_new_weight

# Re-normalize so they sum to 1.0
wide_midfielder_weights = {k: v/total_wm_weight for k, v in wide_midfielder_weights.items()}

In [220]:
print("Wide Midfielder Weights:")
for stat, weight in wide_midfielder_weights.items():
    print(f"{stat}: {weight:.4f}")

Wide Midfielder Weights:
goals_p90: 0.0643
assists_p90: 0.0926
non_goal_shots_p90: 0.0452
shot_accuracy: 0.0175
passes_p90: 0.1483
pass_accuracy: 0.0463
dribbles_p90: 0.0869
dribble_success_rate: 0.0672
tackles_p90: 0.0634
tackle_success_rate: 0.0202
offsides_p90: 0.0346
fouls_committed_p90: 0.0235
possession_won_p90: 0.0610
possession_lost_p90: 0.0349
distance_covered_p90: 0.0677
distance_sprinted_p90: 0.0667
xt_bonus_p90: 0.0595


In [221]:
# Saving these weights back to the position_weights.json file
section_names = ["RM", "LM"]
weights_dict = dict(zip(col_names, wide_midfielder_weights.values()))
position_weights_path = project_root / "workshop" / "ratings_creation" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

for section_name in section_names:
    position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

**Step 3: Estimate means and standard deviations for Wide Midfielders**<br><br>

To estimate the means and standard deviations for each stat for the Wide Midfielder position, we use a similar method to the weights, where we take the means and standard deviations from the Winger position and adjust them based on the expected differences in performance for Wide Midfielders compared to Wingers. For example, we would expect Wide Midfielders to have lower means for attacking stats like goals and shots, and higher means for defensive stats like tackles and possession won. The specific adjustments would be based on football knowledge and assumptions about the typical performance of Wide Midfielders compared to Wingers.

In [222]:
wm_stat_multipliers = {
    'goals_p90': 0.5,
    'assists_p90': 1.3,
    'non_goal_shots_p90': 0.5,
    'passes_p90': 1.4,
    'pass_accuracy': 1.06,
    'dribbles_p90': 0.8,
    'tackles_p90': 2.5,
    'tackle_success_rate': 1.5,
    'offsides_p90': 0.8,
    'fouls_committed_p90': 1.2,
    'possession_won_p90': 2.0,
    'possession_lost_p90': 0.7,
    'distance_covered_p90': 1.1,
    'distance_sprinted_p90': 1.1,
    'xt_bonus_p90': 1.3
}

In [223]:
# Importing the winger means and standard deviations
position_means_stds_path = project_root / "workshop" / "ratings_creation" / "position_means_stds.json"
with open(position_means_stds_path, 'r') as f:
    position_means_stds = json.load(f)
winger_means_stds = position_means_stds["RW"]

In [224]:
for stat, stats in winger_means_stds.items():
    print(f"{stat}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

goals_p90: mean=0.1728, std=0.3238
assists_p90: mean=0.1585, std=0.2956
non_goal_shots_p90: mean=0.4696, std=0.5215
shot_accuracy: mean=32.4120, std=41.0999
passes_p90: mean=17.9666, std=6.5569
pass_accuracy: mean=77.0731, std=20.7384
dribbles_p90: mean=17.4586, std=4.8053
dribble_success_rate: mean=82.0864, std=18.7963
tackles_p90: mean=2.5216, std=1.4783
tackle_success_rate: mean=25.5548, std=37.8703
offsides_p90: mean=0.1930, std=0.2323
fouls_committed_p90: mean=0.0739, std=0.1551
possession_won_p90: mean=0.6412, std=0.4367
possession_lost_p90: mean=1.7825, std=0.3485
distance_covered_p90: mean=11.3249, std=0.7894
distance_sprinted_p90: mean=3.9658, std=1.0869
xt_bonus_p90: mean=0.3191, std=0.2571


In [225]:
wm_means_stds = {}
for stat, stats in winger_means_stds.items():
    mean_multiplier = wm_stat_multipliers.get(stat, 1.0)
    std_multiplier = np.sqrt(mean_multiplier) # Assuming variance scales with mean
    wm_means_stds[stat] = {
        "mean": stats["mean"] * mean_multiplier,
        "std": stats["std"] * std_multiplier
    }

In [226]:
print("Wide Midfielder Means and Standard Deviations:")
for stat, stats in wm_means_stds.items():
    print(f"{stat}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

Wide Midfielder Means and Standard Deviations:
goals_p90: mean=0.0864, std=0.2290
assists_p90: mean=0.2060, std=0.3371
non_goal_shots_p90: mean=0.2348, std=0.3687
shot_accuracy: mean=32.4120, std=41.0999
passes_p90: mean=25.1533, std=7.7582
pass_accuracy: mean=81.6975, std=21.3515
dribbles_p90: mean=13.9669, std=4.2980
dribble_success_rate: mean=82.0864, std=18.7963
tackles_p90: mean=6.3039, std=2.3373
tackle_success_rate: mean=38.3322, std=46.3815
offsides_p90: mean=0.1544, std=0.2078
fouls_committed_p90: mean=0.0887, std=0.1699
possession_won_p90: mean=1.2824, std=0.6176
possession_lost_p90: mean=1.2478, std=0.2916
distance_covered_p90: mean=12.4574, std=0.8279
distance_sprinted_p90: mean=4.3624, std=1.1400
xt_bonus_p90: mean=0.4148, std=0.2932


In [227]:
# Storing the new means and standard deviations back to the position_means_stds.json file
section_names = ["RM", "LM"]
position_means_stds_path = project_root / "workshop" / "ratings_creation" / "position_means_stds.json"
if position_means_stds_path.exists():
    with open(position_means_stds_path, "r") as f:
        position_means_stds = json.load(f)
else:
    position_means_stds = {}

for section_name in section_names:
    position_means_stds[section_name] = wm_means_stds

with open(position_means_stds_path, "w") as f:
    json.dump(position_means_stds, f, indent=4)

## The CAM
**Proxy: Central Midfielder (CM)**<br><br>
The CAM is pure creation and goal threat, with much less defensive responsibility than a traditional central midfielder. Therefore, we will increase the weights for goals, assists, passing, and xT, while decreasing the weights for defensive actions and distance covered. The specific multipliers will be determined based on the typical responsibilities of CAMs in modern football tactics.

**Step 1: Define weight multipliers for Wide Midfielder**
 - `Goals`: **2.0** (CAMs should be much more rewarded for goals than traditional central midfielders, as they are often a team's primary goal threat)
 - `Assists`: **2.5** (similar to goals, CAMs should be more rewarded for assists than traditional central midfielders, as they are often a team's primary creative force)
 - `Shots`: **1.5** (CAMs should be rewarded more for shots than traditional central midfielders due to their increased attacking involvement)
 - `Shot Accuracy`: **1.5** (CAMs should have better shot accuracy due to more attacking involvement)
 - `Passes`: **1.5** (CAMs should have more passing involvement than traditional central midfielders, as they are often heavily involved in build-up play)
 - `Pass Accuracy`: **1.2** (CAMs should have a higher pass accuracy than traditional central midfielders, as they are often making more key passes and may have more time on the ball)
 - `Dribbles`: **1.5** (CAMs should be expected to dribble more than traditional central midfielders, as they often have more freedom to take on players and create chances)
 - `Dribble Success Rate`: **1.2** (CAMs should be rewarded more for successful dribbles than traditional central midfielders, as their dribbles are often more impactful and in tighter spaces)
 - `Tackles`: **0.2** (CAMs have less defensive responsibilities than traditional central midfielders, so they should be expected to make fewer tackles)
 - `Tackle Success Rate`: **0.2** (due to their reduced defensive responsibilities, CAMs may have a much lower tackle success rate than traditional central midfielders)
 - `Offsides`: **1.0** (CAMs may have similar offsides involvement as traditional central midfielders, but this can vary greatly depending on the player's role and style)
 - `Fouls Committed`: **1.0** (CAMs may have similar fouls committed as traditional central midfielders, but this can vary greatly depending on the player's role and style)
 - `Possession Won`: **0.3** (there is a much lower defensive burden on CAMs, so they may win significantly fewer possessions than traditional central midfielders)
 - `Possession Lost`: **0.5** (CAMs should have the freedom to be more creative and take more risks, so they should be punished less for losing possessions than traditional central midfielders)
 - `Distance Covered`: **0.8** (As CAMs have less defensive responsibilities than traditional central midfielders, they may be expected to cover less distance than traditional central midfielders, but this can vary greatly depending on the player's role and style)
 - `Distance Sprinted`: **0.8** (similar to distance covered, CAMs may be expected to sprint less than traditional central midfielders due to their reduced defensive responsibilities, but this can vary greatly depending on the player's role and style)
 - `xT`: **2.0** (CAMs should be contributing much more to expected threat than traditional central midfielders due to their increased attacking involvement)

In [228]:
cam_multipliers = {
    'goals_p90': 2.0,
    'assists_p90': 2.5,
    'non_goal_shots_p90': 1.5,
    'shot_accuracy': 1.5,
    'passes_p90': 1.5,
    'pass_accuracy': 1.2,
    'dribbles_p90': 1.5,
    'dribble_success_rate': 1.2,
    'tackles_p90': 0.2,
    'tackle_success_rate': 0.2,
    'offsides_p90': 1.0,
    'fouls_committed_p90': 1.0,
    'possession_won_p90': 0.3,
    'possession_lost_p90': 0.5,
    'distance_covered_p90': 0.8,
    'distance_sprinted_p90': 0.8,
    'xt_bonus_p90': 2.0
}

In [229]:
# Importing central midfielder weights
col_names = ["goals_p90", "assists_p90", "non_goal_shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
position_weights_path = project_root / "workshop" / "ratings_creation" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["CM"]
cm_weights = {col: weights_dict[col] for col in col_names}

In [230]:
cam_weights = {}
total_cam_weight = 0

for stat, base_weight in cm_weights.items():
    multiplier = cam_multipliers.get(stat, 1.0)
    raw_new_weight = base_weight * multiplier
    cam_weights[stat] = raw_new_weight
    total_cam_weight += raw_new_weight

cam_weights = {k: v/total_cam_weight for k, v in cam_weights.items()}

In [231]:
print("Attacking Midfielder Weights:")
for stat, weight in cam_weights.items():
    print(f"{stat}: {weight:.4f}")

Attacking Midfielder Weights:
goals_p90: 0.1152
assists_p90: 0.1320
non_goal_shots_p90: 0.0973
shot_accuracy: 0.0272
passes_p90: 0.1097
pass_accuracy: 0.0664
dribbles_p90: 0.0978
dribble_success_rate: 0.0525
tackles_p90: 0.0122
tackle_success_rate: 0.0070
offsides_p90: 0.0006
fouls_committed_p90: 0.0347
possession_won_p90: 0.0192
possession_lost_p90: 0.0492
distance_covered_p90: 0.0386
distance_sprinted_p90: 0.0464
xt_bonus_p90: 0.0941


In [232]:
# Saving these weights back to the position_weights.json file
section_names = ["CAM"]
weights_dict = dict(zip(col_names, cam_weights.values()))
position_weights_path = project_root / "workshop" / "ratings_creation" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

for section_name in section_names:
    position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

**Step 3: Estimate means and standard deviations for CAMs**<br><br>

To estimate the means and standard deviations for each stat for the CAM position, we use a similar method to the weights, where we take the means and standard deviations from the Central Midfielder position and adjust them based on the expected differences in performance for CAMs compared to traditional central midfielders. For example, we would expect CAMs to have higher means for attacking stats like goals, assists, and xT, and lower means for defensive stats like tackles and possession won. The specific adjustments would be based on football knowledge and assumptions about the typical performance of CAMs compared to traditional central midfielders.

In [233]:
cam_stat_multipliers = {
    'goals_p90': 1.5,
    'assists_p90': 2.0,
    'non_goal_shots_p90': 1.5,
    'passes_p90': 1.2,
    'pass_accuracy': 0.95,
    'dribbles_p90': 1.2,
    'dribble_success_rate': 0.95,
    'tackles_p90': 0.2,
    'offsides_p90': 1.2,
    'possession_won_p90': 0.3,
    'possession_lost_p90': 1.3,
    'distance_covered_p90': 0.85,
    'distance_sprinted_p90': 0.85,
    'xt_bonus_p90': 1.3
}

In [234]:
# Importing the cm means and standard deviations
position_means_stds_path = project_root / "workshop" / "ratings_creation" / "position_means_stds.json"
with open(position_means_stds_path, 'r') as f:
    position_means_stds = json.load(f)
cm_means_stds = position_means_stds["CM"]

In [235]:
for stat, stats in cm_means_stds.items():
    print(f"{stat}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

goals_p90: mean=0.1380, std=0.2946
assists_p90: mean=0.1463, std=0.2776
non_goal_shots_p90: mean=0.4709, std=0.4467
shot_accuracy: mean=31.2191, std=40.5874
passes_p90: mean=33.5830, std=7.1616
pass_accuracy: mean=88.1873, std=10.2208
dribbles_p90: mean=28.2992, std=7.4282
dribble_success_rate: mean=91.0601, std=9.2920
tackles_p90: mean=5.2634, std=2.1155
tackle_success_rate: mean=26.8905, std=30.8306
offsides_p90: mean=0.0285, std=0.1199
fouls_committed_p90: mean=0.1520, std=0.2249
possession_won_p90: mean=1.2427, std=0.4857
possession_lost_p90: mean=1.7547, std=0.3988
distance_covered_p90: mean=12.0294, std=1.0410
distance_sprinted_p90: mean=4.5731, std=1.3279
xt_bonus_p90: mean=0.5398, std=0.3895


In [236]:
cam_means_stds = {}
for stat, stats in cm_means_stds.items():
    mean_multiplier = cam_stat_multipliers.get(stat, 1.0)
    std_multiplier = np.sqrt(mean_multiplier) # Assuming variance scales with mean
    cam_means_stds[stat] = {
        "mean": stats["mean"] * mean_multiplier,
        "std": stats["std"] * std_multiplier
    }

In [237]:
print("Attacking Midfielder Means and Standard Deviations:")
for stat, stats in cam_means_stds.items():
    print(f"{stat}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

Attacking Midfielder Means and Standard Deviations:
goals_p90: mean=0.2069, std=0.3608
assists_p90: mean=0.2926, std=0.3926
non_goal_shots_p90: mean=0.7063, std=0.5471
shot_accuracy: mean=31.2191, std=40.5874
passes_p90: mean=40.2996, std=7.8451
pass_accuracy: mean=83.7779, std=9.9620
dribbles_p90: mean=33.9591, std=8.1372
dribble_success_rate: mean=86.5071, std=9.0567
tackles_p90: mean=1.0527, std=0.9461
tackle_success_rate: mean=26.8905, std=30.8306
offsides_p90: mean=0.0342, std=0.1313
fouls_committed_p90: mean=0.1520, std=0.2249
possession_won_p90: mean=0.3728, std=0.2660
possession_lost_p90: mean=2.2812, std=0.4547
distance_covered_p90: mean=10.2250, std=0.9598
distance_sprinted_p90: mean=3.8872, std=1.2243
xt_bonus_p90: mean=0.7018, std=0.4441


In [238]:
# Storing the new means and standard deviations back to the position_means_stds.json file
section_names = ["CAM"]
position_means_stds_path = project_root / "workshop" / "ratings_creation" / "position_means_stds.json"
if position_means_stds_path.exists():
    with open(position_means_stds_path, "r") as f:
        position_means_stds = json.load(f)
else:
    position_means_stds = {}

for section_name in section_names:
    position_means_stds[section_name] = cam_means_stds

with open(position_means_stds_path, "w") as f:
    json.dump(position_means_stds, f, indent=4)